# Few-shot prompting test — Qwen2.5-Coder-7B

**Goal:** before spending $5-10 on training, see if clever prompting + in-context examples can already get the 7B model to produce path-based letter shapes. If yes → training will be easy; if no → training has more work.

**How:** for 5 fresh test prompts, generate at 3 shot counts:
- **0-shot**: bare prompt (matches our baseline run)
- **1-shot**: one example before target (Roboto 'O')
- **3-shot**: three examples covering sans + serif + handwriting

5 prompts × 3 shot counts = 15 generations. ~10 min total on A100.

**How to use:** Runtime → Change runtime type → A100, then Runtime → Run all.

In [ ]:
!pip install -q 'transformers>=4.42' 'torch>=2.1' 'accelerate>=0.30' 'cairosvg>=2.7'

In [ ]:
import json, sys, xml.etree.ElementTree as ET
from pathlib import Path
import torch
import cairosvg
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'Qwen/Qwen2.5-Coder-7B'
OUT_DIR = Path('/content/fewshot')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INSTRUCTION_PREFIX = 'Generate an SVG glyph for: '
RESPONSE_TEMPLATE = '\nSVG:\n'
MAX_NEW_TOKENS = 2048

# Three in-context examples pulled from our actual training data.
# Real (caption, svg) pairs from glyphs.parquet — Roboto O, Lora A, Pacifico h.
FEW_SHOT_EXAMPLES = [
    {
        'caption': "the letter 'O' in a sans-serif typeface; neo-grotesque, business, competent, calm",
        'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 1000"><path d="M538 469V508Q538 576 520 630Q503 684 471 722Q439 760 394 780Q350 800 294 800Q240 800 196 780Q151 760 118 722Q85 684 68 630Q50 576 50 508V469Q50 401 67 347Q85 293 118 255Q150 217 195 197Q240 177 293 177Q349 177 394 197Q438 217 471 255Q503 293 520 347Q538 401 538 469ZM458 508V468Q458 414 447 373Q436 331 415 303Q393 274 363 260Q332 245 293 245Q256 245 226 260Q195 274 174 303Q152 331 141 373Q129 414 129 468V508Q129 562 141 604Q152 646 174 674Q196 703 226 718Q257 732 294 732Q333 732 364 718Q394 703 415 674Q436 646 447 604Q458 562 458 508Z"/></svg>'
    },
    {
        'caption': "the letter 'A' in a serif typeface; business, competent, sincere",
        'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 1000"><path d="M-5 786V760Q19 760 31 748Q44 735 52 713Q60 691 71 662L232 224H251L425 669Q431 685 439 711Q447 737 445 755Q459 754 472 753Q485 752 498 752V786H320V760Q349 759 359 747Q368 734 366 717Q363 700 358 686L330 612L127 617L100 701Q95 720 90 730Q86 741 80 755Q96 754 111 753Q127 752 141 752V786ZM139 584H320L264 434Q255 410 246 386Q238 362 229 338H227Q220 360 211 382Q203 404 195 427Z"/></svg>'
    },
    {
        'caption': "the letter 'h' in a handwriting typeface; informal",
        'svg': '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 1000"><path d="M359 641Q359 663 349 676Q322 706 295 726Q267 745 232 745Q203 745 188 728Q173 711 173 679Q173 663 181 622Q188 587 188 573Q188 564 182 564Q175 564 161 583Q147 602 134 633Q120 665 112 699Q101 745 58 745Q41 745 36 733Q30 720 30 688Q30 670 31 659L31 614Q31 526 49 431Q67 336 102 271Q137 207 186 207Q212 207 229 229Q245 252 245 288Q245 345 211 407Q178 469 102 551Q100 581 100 612Q119 564 142 533Q165 503 188 490Q210 477 229 477Q266 477 266 514Q266 536 253 594Q243 644 243 659Q243 682 259 682Q271 682 286 668Q302 654 328 623Q335 615 343 615Q351 615 355 622Q359 629 359 641ZM108 480Q144 438 167 390Q190 341 190 301Q190 282 186 273Q182 263 174 263Q164 263 151 292Q139 322 127 372Q116 421 108 480Z"/></svg>'
    },
]

# Test prompts — none overlap with the few-shot examples
TEST_PROMPTS = [
    ("the letter 'B' in a sans-serif typeface; humanist, calm", 'B'),
    ("the letter 'M' in a serif typeface; transitional, formal", 'M'),
    ("the letter 'p' in a handwriting typeface; informal, cursive", 'p'),
    ("the letter 'X' in a monospace typeface; technical, clean", 'X'),
    ("the letter 'g' in a display typeface; playful, rounded", 'g'),
]

SHOT_COUNTS = [0, 1, 3]
print(f'{len(TEST_PROMPTS)} test prompts × {len(SHOT_COUNTS)} variants = {len(TEST_PROMPTS) * len(SHOT_COUNTS)} generations')

In [ ]:
print(f'Loading {MODEL}...')
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='auto')
model.eval()
print(f'Loaded. Device: {model.device}, dtype: {model.dtype}')

In [ ]:
def extract_svg(text):
    s = text.find('<svg')
    if s < 0: return None
    e = text.find('</svg>', s)
    if e < 0: return None
    return text[s:e + len('</svg>')]

def is_valid_xml(s):
    try:
        ET.fromstring(s); return True
    except ET.ParseError:
        return False

def uses_text_shortcut(svg):
    return svg is not None and '<text' in svg

def build_prompt(target_caption, n_shots):
    parts = []
    for ex in FEW_SHOT_EXAMPLES[:n_shots]:
        parts.append(f'{INSTRUCTION_PREFIX}{ex["caption"]}{RESPONSE_TEMPLATE}{ex["svg"]}')
    parts.append(f'{INSTRUCTION_PREFIX}{target_caption}{RESPONSE_TEMPLATE}')
    return '\n\n'.join(parts)

In [ ]:
results = []
for n_shots in SHOT_COUNTS:
    print(f'\n=== {n_shots}-shot ===', flush=True)
    shot_dir = OUT_DIR / f'shots_{n_shots}'
    shot_dir.mkdir(exist_ok=True)
    for i, (caption, target) in enumerate(TEST_PROMPTS):
        prompt = build_prompt(caption, n_shots)
        inputs = tok(prompt, return_tensors='pt').to(model.device)
        prompt_tokens = inputs.input_ids.shape[1]
        with torch.no_grad():
            out_ids = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, pad_token_id=tok.pad_token_id,
            )
        new = out_ids[0][prompt_tokens:]
        gen = tok.decode(new, skip_special_tokens=True)
        svg = extract_svg(gen)
        valid_xml = bool(svg) and is_valid_xml(svg)
        used_text = uses_text_shortcut(svg)
        rendered = False
        (shot_dir / f'sample_{i:02d}_raw.txt').write_text(gen)
        if svg:
            (shot_dir / f'sample_{i:02d}.svg').write_text(svg)
            try:
                cairosvg.svg2png(bytestring=svg.encode(),
                                 write_to=str(shot_dir / f'sample_{i:02d}.png'),
                                 output_width=128, output_height=128)
                rendered = True
            except Exception:
                pass
        flags = '+'.join(x for x, ok in [
            ('svg', svg), ('xml', valid_xml), ('png', rendered)] if ok) or '—'
        text_flag = '  ⚠ <text>' if used_text else ''
        print(f'  [{i}] {target!r:>4}  ({flags:>11})  prompt_tokens={prompt_tokens:>5}  gen={int(new.shape[0]):>4}{text_flag}  caption={caption[:50]}...', flush=True)
        results.append({
            'shots': n_shots, 'i': i, 'caption': caption, 'target': target,
            'prompt_tokens': int(prompt_tokens),
            'n_gen_tokens': int(new.shape[0]),
            'svg_extracted': svg is not None,
            'valid_xml': valid_xml,
            'used_text_shortcut': used_text,
            'rendered': rendered,
        })
(OUT_DIR / '_summary.json').write_text(json.dumps(results, indent=2))

In [ ]:
# Per-shot summary
print('=== Per-shot summary ===')
for n in SHOT_COUNTS:
    rs = [r for r in results if r['shots'] == n]
    nr = sum(r['rendered'] for r in rs)
    nt = sum(r['used_text_shortcut'] for r in rs)
    np_ = sum(r['rendered'] and not r['used_text_shortcut'] for r in rs)
    print(f'  {n}-shot: rendered={nr}/{len(rs)},  via <text>={nt},  via <path>-only={np_}')
print('\nKey question: does N-shot produce <path>-only letter shapes that 0-shot didn\'t?')
print('Inspect the rendered samples below — the visual is what matters, not just counts.')

# Display all rendered images grouped by shot count
from IPython.display import display, Image, Markdown
for n in SHOT_COUNTS:
    display(Markdown(f'## {n}-shot results'))
    for i, (caption, target) in enumerate(TEST_PROMPTS):
        p = OUT_DIR / f'shots_{n}' / f'sample_{i:02d}.png'
        r = next((x for x in results if x['shots']==n and x['i']==i), None)
        if p.exists():
            text_note = ' (used `<text>` shortcut)' if r and r['used_text_shortcut'] else ''
            display(Markdown(f'**target=`{target}`**{text_note} — {caption[:70]}'))
            display(Image(filename=str(p)))

In [ ]:
# Download all results as a zip
import shutil
from google.colab import files
shutil.make_archive('/content/fewshot', 'zip', '/content/fewshot')
files.download('/content/fewshot.zip')